# Barrido de λ (soft-clDice)

**Objetivo:** estudiar si aumentar el peso de soft-clDice mejora la detección (cobertura y clDice) a costa del Dice.

Se sigue el patrón validado en el notebook 12 (v1): el código del trainer se escribe en un archivo `.py` dentro de `custom_trainers/`, esa carpeta se registra con la variable de entorno `nnUNet_extTrainer` y el entrenamiento se lanza con `nnUNetv2_train -tr <NombreDeClase>`.

Para cada λ solo varían un número dentro del archivo `.py` generado (`self.lambda_cldice = ...`) y el nombre de la clase, de modo que cada configuración se almacena en su propia carpeta de resultados.

Modelos ya entrenados, que no se repiten:
- λ = 0.0 → baseline
- λ = 0.2 → v1 (notebook 12)

Los valores de λ entrenados en este barrido son los definidos en `LAMBDAS_TO_RUN` (celda de configuración).


## 1. Entorno: montaje de Drive y variables de nnU-Net (igual que en el notebook 12)


In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA disponible: True
GPU: NVIDIA A100-SXM4-40GB


In [2]:
!python -m pip install -q nnunetv2

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.1/291.1 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.8/76.8 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.1/28.1 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 98.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 115.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 9.9 MB/s eta 0:00:00


In [3]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
import os
from pathlib import Path

BASE = Path("/content/drive/MyDrive/TFM_nnUNet")

NNUNET_RAW = BASE / "nnUNet_raw"
NNUNET_PREPROCESSED = BASE / "nnUNet_preprocessed"
NNUNET_RESULTS = BASE / "nnUNet_results"

os.environ["nnUNet_raw"] = str(NNUNET_RAW)
os.environ["nnUNet_preprocessed"] = str(NNUNET_PREPROCESSED)
os.environ["nnUNet_results"] = str(NNUNET_RESULTS)

print("RAW:", NNUNET_RAW)
print("PREPROCESSED:", NNUNET_PREPROCESSED)
print("RESULTS:", NNUNET_RESULTS)

RAW: /content/drive/MyDrive/TFM_nnUNet/nnUNet_raw
PREPROCESSED: /content/drive/MyDrive/TFM_nnUNet/nnUNet_preprocessed
RESULTS: /content/drive/MyDrive/TFM_nnUNet/nnUNet_results


In [5]:
# Carpeta de trainers personalizados, la misma que en el notebook 12
CUSTOM_TRAINERS = BASE / "custom_trainers"
CUSTOM_TRAINERS.mkdir(parents=True, exist_ok=True)

os.environ["nnUNet_extTrainer"] = str(CUSTOM_TRAINERS)

print("Carpeta de trainers:", CUSTOM_TRAINERS)
print("nnUNet_extTrainer:", os.environ["nnUNet_extTrainer"])

Carpeta de trainers: /content/drive/MyDrive/TFM_nnUNet/custom_trainers
nnUNet_extTrainer: /content/drive/MyDrive/TFM_nnUNet/custom_trainers


## 2. Configuración fija del barrido

`DATASET_ID = 501` corresponde a `Dataset501_ImageCAS`, carpeta de `nnUNet_preprocessed/` con los 200 casos del fold 0, la misma que utilizan baseline, v1 y v2.


In [6]:
DATASET_ID = 501         # Dataset501_ImageCAS
CONFIGURATION = "3d_fullres"
FOLD = 0
NUM_EPOCHS = 100         # mismo que baseline/v1/v2, para comparabilidad

# Valores de λ entrenados en este barrido (se excluyen los ya entrenados: 0.0 y 0.2)
LAMBDAS_TO_RUN = [0.1, 0.4, 0.8]

## 3. Plantilla del trainer (idéntica a la del notebook 12, con λ parametrizado)

La celda define una función que genera el mismo código empleado en v1, modificando únicamente `self.lambda_cldice` y el nombre de la clase. No se reescribe la lógica de la pérdida: es una copia literal del archivo de v1 con dos campos sustituidos para cada valor de λ.


In [7]:
def generar_codigo_trainer(lam, num_epochs=100):
    lam_str = str(lam).replace(".", "")
    class_name = f"nnUNetTrainerSoftClDice_lam{lam_str}_{num_epochs}epochs"

    codigo = f'''
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from nnunetv2.training.loss.compound_losses import DC_and_CE_loss
from nnunetv2.training.loss.deep_supervision import DeepSupervisionWrapper
from nnunetv2.training.loss.dice import MemoryEfficientSoftDiceLoss

from nnunetv2.training.nnUNetTrainer.variants.training_length.nnUNetTrainer_Xepochs import (
    nnUNetTrainer_100epochs
)


def soft_erode_3d(x):

    p1 = -F.max_pool3d(
        -x,
        kernel_size=(3, 1, 1),
        stride=1,
        padding=(1, 0, 0)
    )

    p2 = -F.max_pool3d(
        -x,
        kernel_size=(1, 3, 1),
        stride=1,
        padding=(0, 1, 0)
    )

    p3 = -F.max_pool3d(
        -x,
        kernel_size=(1, 1, 3),
        stride=1,
        padding=(0, 0, 1)
    )

    return torch.minimum(
        torch.minimum(p1, p2),
        p3
    )


def soft_dilate_3d(x):

    return F.max_pool3d(
        x,
        kernel_size=3,
        stride=1,
        padding=1
    )


def soft_open_3d(x):

    return soft_dilate_3d(
        soft_erode_3d(x)
    )


def soft_skeletonize_3d(x, iterations=10):

    opened = soft_open_3d(x)

    skeleton = F.relu(
        x - opened
    )

    for _ in range(iterations):

        x = soft_erode_3d(x)

        opened = soft_open_3d(x)

        delta = F.relu(
            x - opened
        )

        skeleton = (
            skeleton
            + F.relu(
                delta - skeleton * delta
            )
        )

    return skeleton


class SoftClDiceLoss(nn.Module):

    def __init__(
        self,
        iterations=10,
        smooth=1e-5
    ):
        super().__init__()

        self.iterations = iterations
        self.smooth = smooth


    def forward(
        self,
        prediction,
        target
    ):

        prediction = prediction.float()
        target = target.float()

        skel_pred = soft_skeletonize_3d(
            prediction,
            iterations=self.iterations
        )

        skel_true = soft_skeletonize_3d(
            target,
            iterations=self.iterations
        )

        tprec = (
            (skel_pred * target).sum()
            + self.smooth
        ) / (
            skel_pred.sum()
            + self.smooth
        )

        tsens = (
            (skel_true * prediction).sum()
            + self.smooth
        ) / (
            skel_true.sum()
            + self.smooth
        )

        cldice = (
            2.0 * tprec * tsens
        ) / (
            tprec
            + tsens
            + self.smooth
        )

        return 1.0 - cldice


class DiceCEPlusSoftClDice(nn.Module):

    def __init__(
        self,
        regional_loss,
        lambda_cldice={lam},
        skeleton_iterations=10
    ):
        super().__init__()

        self.regional_loss = regional_loss
        self.lambda_cldice = lambda_cldice

        self.soft_cldice = SoftClDiceLoss(
            iterations=skeleton_iterations,
            smooth=1e-5
        )


    def forward(
        self,
        logits,
        target
    ):

        loss_regional = self.regional_loss(
            logits,
            target
        )

        probabilities = torch.softmax(
            logits.float(),
            dim=1
        )

        foreground_probability = (
            probabilities[:, 1:2]
        )

        if target.ndim == logits.ndim - 1:
            target = target.unsqueeze(1)

        foreground_target = (
            target == 1
        ).float()

        topology_loss = self.soft_cldice(
            foreground_probability,
            foreground_target
        )

        return (
            loss_regional
            +
            self.lambda_cldice
            * topology_loss
        )


class {class_name}(
    nnUNetTrainer_100epochs
):

    def __init__(
        self,
        plans,
        configuration,
        fold,
        dataset_json,
        device=torch.device("cuda")
    ):

        super().__init__(
            plans,
            configuration,
            fold,
            dataset_json,
            device
        )

        self.lambda_cldice = {lam}
        self.skeleton_iterations = 10


    def _build_loss(self):

        regional_loss = DC_and_CE_loss(

            soft_dice_kwargs={{
                "batch_dice":
                    self.configuration_manager.batch_dice,

                "smooth": 1e-5,

                "do_bg": False,

                "ddp": self.is_ddp
            }},

            ce_kwargs={{}},

            weight_ce=1,

            weight_dice=1,

            ignore_label=
                self.label_manager.ignore_label,

            dice_class=
                MemoryEfficientSoftDiceLoss
        )


        loss = DiceCEPlusSoftClDice(

            regional_loss=
                regional_loss,

            lambda_cldice=
                self.lambda_cldice,

            skeleton_iterations=
                self.skeleton_iterations
        )


        if self.enable_deep_supervision:

            deep_supervision_scales = (
                self._get_deep_supervision_scales()
            )

            weights = np.array([
                1 / (2 ** i)
                for i in range(
                    len(deep_supervision_scales)
                )
            ])

            weights[-1] = 0

            weights = (
                weights
                / weights.sum()
            )

            loss = DeepSupervisionWrapper(
                loss,
                weights
            )


        return loss
'''
    return class_name, codigo


# Prueba rápida: generación (sin guardar) del código para λ = 0.4 y comprobación
# de que el nombre de la clase y el valor se han insertado correctamente
_nombre_prueba, _codigo_prueba = generar_codigo_trainer(0.4)
print("Nombre de clase generado:", _nombre_prueba)
assert "self.lambda_cldice = 0.4" in _codigo_prueba
assert f"class {_nombre_prueba}(" in _codigo_prueba
print("OK: la plantilla inserta correctamente el valor de λ y el nombre de clase.")

Nombre de clase generado: nnUNetTrainerSoftClDice_lam04_100epochs
OK: la plantilla inserta correctamente el valor de λ y el nombre de clase.


## 4. Verificación con datos sintéticos

Al igual que el test de gradientes del notebook 12, antes del entrenamiento real se comprueba, con tensores aleatorios en lugar de datos reales, que la pérdida combinada responde correctamente a distintos valores de λ.


In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# Copia de las piezas de pérdida del archivo del trainer v1 (sin el trainer de
# nnU-Net, innecesario aquí), para poder probarlas de forma aislada
def soft_erode_3d(x):
    p1 = -F.max_pool3d(-x, kernel_size=(3, 1, 1), stride=1, padding=(1, 0, 0))
    p2 = -F.max_pool3d(-x, kernel_size=(1, 3, 1), stride=1, padding=(0, 1, 0))
    p3 = -F.max_pool3d(-x, kernel_size=(1, 1, 3), stride=1, padding=(0, 0, 1))
    return torch.minimum(torch.minimum(p1, p2), p3)


def soft_dilate_3d(x):
    return F.max_pool3d(x, kernel_size=3, stride=1, padding=1)


def soft_open_3d(x):
    return soft_dilate_3d(soft_erode_3d(x))


def soft_skeletonize_3d(x, iterations=10):
    opened = soft_open_3d(x)
    skeleton = F.relu(x - opened)
    for _ in range(iterations):
        x = soft_erode_3d(x)
        opened = soft_open_3d(x)
        delta = F.relu(x - opened)
        skeleton = skeleton + F.relu(delta - skeleton * delta)
    return skeleton


class SoftClDiceLoss(nn.Module):
    def __init__(self, iterations=10, smooth=1e-5):
        super().__init__()
        self.iterations = iterations
        self.smooth = smooth

    def forward(self, prediction, target):
        prediction = prediction.float()
        target = target.float()
        skel_pred = soft_skeletonize_3d(prediction, iterations=self.iterations)
        skel_true = soft_skeletonize_3d(target, iterations=self.iterations)
        tprec = ((skel_pred * target).sum() + self.smooth) / (skel_pred.sum() + self.smooth)
        tsens = ((skel_true * prediction).sum() + self.smooth) / (skel_true.sum() + self.smooth)
        cldice = (2.0 * tprec * tsens) / (tprec + tsens + self.smooth)
        return 1.0 - cldice


class DiceCEPlusSoftClDice(nn.Module):
    def __init__(self, regional_loss, lambda_cldice=0.2, skeleton_iterations=10):
        super().__init__()
        self.regional_loss = regional_loss
        self.lambda_cldice = lambda_cldice
        self.soft_cldice = SoftClDiceLoss(iterations=skeleton_iterations, smooth=1e-5)

    def forward(self, logits, target):
        loss_regional = self.regional_loss(logits, target)
        probabilities = torch.softmax(logits.float(), dim=1)
        foreground_probability = probabilities[:, 1:2]
        if target.ndim == logits.ndim - 1:
            target = target.unsqueeze(1)
        foreground_target = (target == 1).float()
        topology_loss = self.soft_cldice(foreground_probability, foreground_target)
        return loss_regional + self.lambda_cldice * topology_loss


torch.manual_seed(0)

logits = torch.randn(2, 2, 16, 16, 16)
target = torch.randint(0, 2, (2, 1, 16, 16, 16)).float()


class _DummyRegionalLoss(nn.Module):
    def forward(self, logits, target):
        return torch.tensor(1.0)


regional = _DummyRegionalLoss()

print("λ  ->  loss total (regional fijo=1.0 + λ·topología)")
valores = []
for lam in [0.0, 0.1, 0.2, 0.4, 0.8]:
    combined = DiceCEPlusSoftClDice(regional_loss=regional, lambda_cldice=lam, skeleton_iterations=10)
    with torch.no_grad():
        loss_value = combined(logits, target).item()
    valores.append(loss_value)
    print(f"{lam:>4}  ->  {loss_value:.4f}")

assert len(set(round(v, 6) for v in valores)) > 1, "El loss no varía con λ — revisa el mecanismo antes de continuar"
print("\nOK: el loss total varía con λ como se espera.")

λ  ->  loss total (regional fijo=1.0 + λ·topología)
 0.0  ->  1.0000
 0.1  ->  1.0503
 0.2  ->  1.1006
 0.4  ->  1.2012
 0.8  ->  1.4023

OK: el loss total varía con λ como se espera.


## 5. Generación de un archivo `.py` por cada λ

Procedimiento equivalente al de registro del trainer del notebook 12.


In [9]:
rutas_generadas = {}

for lam in LAMBDAS_TO_RUN:
    class_name, codigo = generar_codigo_trainer(lam, num_epochs=NUM_EPOCHS)
    trainer_file = CUSTOM_TRAINERS / f"{class_name}.py"
    trainer_file.write_text(codigo)
    rutas_generadas[lam] = (class_name, trainer_file)
    print(f"λ={lam}  ->  {class_name}  guardado en {trainer_file}")

λ=0.1  ->  nnUNetTrainerSoftClDice_lam01_100epochs  guardado en /content/drive/MyDrive/TFM_nnUNet/custom_trainers/nnUNetTrainerSoftClDice_lam01_100epochs.py
λ=0.3  ->  nnUNetTrainerSoftClDice_lam03_100epochs  guardado en /content/drive/MyDrive/TFM_nnUNet/custom_trainers/nnUNetTrainerSoftClDice_lam03_100epochs.py
λ=0.5  ->  nnUNetTrainerSoftClDice_lam05_100epochs  guardado en /content/drive/MyDrive/TFM_nnUNet/custom_trainers/nnUNetTrainerSoftClDice_lam05_100epochs.py
λ=0.8  ->  nnUNetTrainerSoftClDice_lam08_100epochs  guardado en /content/drive/MyDrive/TFM_nnUNet/custom_trainers/nnUNetTrainerSoftClDice_lam08_100epochs.py


## 6. Entrenamiento de cada λ con reanudación ante interrupciones

Como en el notebook 12, el entrenamiento se lanza con `nnUNetv2_train`. Se utiliza `subprocess` en lugar del comando `!nnUNetv2_train ...` para poder comprobar, dentro de un bucle, si ya existe un checkpoint y, en tal caso, omitir o reanudar el entrenamiento.


In [ ]:
import subprocess

DATASET_FOLDER_NAME = f"Dataset{DATASET_ID:03d}_ImageCAS"

def carpeta_resultados(class_name):
    return (
        NNUNET_RESULTS
        / DATASET_FOLDER_NAME
        / f"{class_name}__nnUNetPlans__{CONFIGURATION}"
        / f"fold_{FOLD}"
    )

resumen = {}

for lam, (class_name, _) in rutas_generadas.items():
    out_folder = carpeta_resultados(class_name)
    checkpoint_final = out_folder / "checkpoint_final.pth"
    checkpoint_latest = out_folder / "checkpoint_latest.pth"

    if checkpoint_final.exists():
        print(f"[λ={lam}] Ya entrenado por completo, se omite ({class_name}).")
        resumen[lam] = "ya_entrenado"
        continue

    cmd = [
        "nnUNetv2_train",
        str(DATASET_ID),
        CONFIGURATION,
        str(FOLD),
        "-tr", class_name,
        "--npz",
    ]
    if checkpoint_latest.exists():
        print(f"[λ={lam}] Checkpoint parcial encontrado, reanudando con --c...")
        cmd.append("--c")

    print(f"--- Entrenando λ={lam} ({class_name}) ---")
    print(" ".join(cmd))

    resultado = subprocess.run(cmd)

    if resultado.returncode == 0:
        resumen[lam] = "completado"
    else:
        resumen[lam] = f"error (código {resultado.returncode})"
        print(f"[λ={lam}] ERROR — revisar el log anterior antes de continuar con el siguiente λ.")

print("\nResumen del barrido:")
for lam, status in resumen.items():
    print(f"  λ={lam}: {status}")

--- Entrenando λ=0.1 (nnUNetTrainerSoftClDice_lam01_100epochs) ---
nnUNetv2_train 501 3d_fullres 0 -tr nnUNetTrainerSoftClDice_lam01_100epochs --npz


## 7. Integración de resultados

Una vez completado el barrido, los resultados de las nuevas carpetas (`nnUNetTrainerSoftClDice_lam*`) se integran con los de baseline, v1 y v2 en el notebook 19, que genera la tabla única con λ como variable continua (Dice, clDice, cobertura de la componente principal y HD95 para cada valor). Esa tabla permite analizar directamente el compromiso entre detección y Dice.
